# 6.1 Shape descriptors

The main objective of region description is to represent a segmented image region with a compact **feature vector** $\mathbf{f}=[f_1,\dots,f_D]^\top \in \mathbb{R}^D$, which preserves the information needed for matching, recognition and classification. This representation can describe the **shape** of the region (using its binary mask or its boundary) or its **appearance** (the intensity, color and texture of its pixels), as shown in Fig. 1.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/region_descriptor_overview.png" width="650"/>
    <figcaption>Fig. 1. A segmented region can be described by its shape (binary mask or boundary contour) and by its appearance (pixel values, texture, color). Both lead to a feature vector $\mathbf{f} \in \mathbb{R}^D$.</figcaption>
</center>$\\[5pt]$

In this notebook we will focus on **shape representation**, which describes the geometry of the region using only its binary mask or its boundary (e.g. its area, perimeter, circularity or elongation). The next notebook covers appearance.

Depending on the application, the descriptor may need to be **invariant** to the position of the region in the image (translation), its orientation (rotation) and/or its size (scale), as illustrated in Fig. 2.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/invariances.png" width="650"/>
    <figcaption>Fig. 2. Invariance to translation, rotation and scale: the descriptor should not change when the region is moved, rotated or resized.</figcaption>
</center>$\\[5pt]$

This notebook covers:

- Simple geometric descriptors: circularity (<a href="#611">section 6.1.1</a>) and elongation (<a href="#612">section 6.1.2</a>), and how to combine them into a feature vector (<a href="#613">section 6.1.3</a>).
- Moments (<a href="#614">section 6.1.4</a>) and Hu moments, a shape descriptor invariant to translation, scale and rotation (<a href="#615">section 6.1.5</a>).

Let's go!

## Problem context - Number-plate recognition

So here we are again! UMA called for us to join a team working on their parking access system. This time, they want to upgrade their obsolete number-plate recognition algorithm with better and more efficient methods.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/access_system.png" width="400"/>
    <figcaption>Fig. 3. UMA's parking access system.</figcaption>
</center>$\\[5pt]$

Here is where our work starts: we are going to **apply shape analysis to each of the characters** that can appear on a license plate, that is, the numbers from 0 to 9 and the letters of the alphabet. The idea is to **produce a unique feature vector** for each character that could appear on a plate (e.g. $\mathbf{f}^0$, $\mathbf{f}^1$, $\dots$, $\mathbf{f}^A$, $\mathbf{f}^B$, etc.), so it can later be used to **train an automatic classification system** (we will see this in the next chapter!).

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Shape descriptors in practice</font>  
> **Suggested prompt**  
> *In the context of computer vision, in which applications are handcrafted shape descriptors (e.g. circularity, elongation or Hu moments) still used today, and when are learned representations from neural networks preferred instead?*

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 06. Region Description'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (15.0, 8.0)

images_path = './images/'

### Initial data

UMA's parking security team has sent us some segmented plate characters captured by their camera in the parking lot. They have binarized and cropped these images, so the regions representing such characters are white pixels. These cropped images are `region_0.png` (region with a zero), `region_6.png` (region with a six), `region_B.png` (region with a B), and `region_J.png` (region with a J).

Let's visualize them!

In [ ]:
# Read the images
zero = cv2.imread(images_path + 'region_0.png', cv2.IMREAD_GRAYSCALE)
J = cv2.imread(images_path + 'region_J.png', cv2.IMREAD_GRAYSCALE)
B = cv2.imread(images_path + 'region_B.png', cv2.IMREAD_GRAYSCALE)
six = cv2.imread(images_path + 'region_6.png', cv2.IMREAD_GRAYSCALE)

# And show them!
plt.subplot(141)
plt.imshow(zero, cmap='gray')
plt.title('Zero region')

plt.subplot(142)
plt.imshow(J, cmap='gray')
plt.title('J region')

plt.subplot(143)
plt.imshow(B, cmap='gray')
plt.title('B region')

plt.subplot(144)
plt.imshow(six, cmap='gray')
plt.title('Six region');

## 6.1.1 Circularity <a id="611"></a>

The first feature we are going to work with is **circularity**, which measures how close a shape is to a circle using its area $A$ and its perimeter $P$ (see Fig. 4):

$$C = \frac{4\pi A}{P^2}$$

A perfect circle has the maximum circularity, $C = 1$ (its area is $A = \pi r^2$ and its perimeter $P = 2\pi r$), while less circular shapes, e.g. elongated regions or regions with an irregular boundary, have smaller values of $C$.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/circularity.png" width="400"/>
    <figcaption>Fig. 4. Circularity relates the area $A$ of a region to its perimeter $P$.</figcaption>
</center>$\\[5pt]$

So, how do we compute $A$ and $P$ for our plate characters? Since they are binary images, the usual approach is to first extract the **contour** of the region, that is, the list of points along its boundary. Then, the perimeter $P$ is the length of that contour, and the area $A$ is the area it encloses. Let's see which OpenCV functions do each of these steps.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Contours in OpenCV</font>  
> OpenCV extracts the contours of a region with [`cv2.findContours()`](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#gadf1ad6a0b82947fa1fe3c3d497f260e0), so let's see how it works!

> **Suggested prompt**  
> *I need to extract the external contour of a binary region with OpenCV's `cv2.findContours()`. Do not write the full solution. Explain what this function takes as input and what it returns, the difference between the retrieval modes `RETR_EXTERNAL`, `RETR_LIST` and `RETR_CCOMP`, and between the approximation methods `CHAIN_APPROX_NONE` and `CHAIN_APPROX_SIMPLE`.*

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Area and perimeter of a contour</font>  
> Once we have the contour, OpenCV also gives us the **area** $A$ and the **perimeter** $P$ of the region with [`cv2.contourArea()`](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#ga2c759ed9f497d4a618048a2f56dc97f1) and [`cv2.arcLength()`](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#ga8d26483c636be6b35c3ec6335798a47c).

> **Suggested prompt**  
> *Given a contour returned by OpenCV's `cv2.findContours()`, I need to compute its area and its perimeter with `cv2.contourArea()` and `cv2.arcLength()`. Do not write the full solution. Explain what each function returns, what the `closed` parameter of `cv2.arcLength()` means, and why the area of the contour may differ from the number of pixels of the region.*

### **<span style="color:green"><b><i>ASSIGNMENT 1: Computing circularity</i></b></span>**

**What to do?** Complete the function `circularity()`, which computes the circularity $C$ of an input region: get its contour with `cv2.findContours()`, and then its area $A$ and perimeter $P$ with `cv2.contourArea()` and `cv2.arcLength()`.

Some characters, like `0`, `B` or `6`, have **holes**, so their boundary is made of an external contour and one internal contour per hole. For simplicity, we are going to take into account **only the external boundary**, so $A$ is the area it encloses (holes included) and $P$ is its length. In other words, we describe the silhouette of the character as if it had no holes.

*Note: Use `cv2.RETR_EXTERNAL` and `cv2.CHAIN_APPROX_NONE`.*

In [ ]:
# Assignment 1
def circularity(region):
    """ Compute the circularity of a region.

        Args:
            region: Binary image

        Returns:
            C: Circularity of the region (between 0 and 1)
    """
    plt.imshow(region,cmap='gray')
    plt.show()
    # Get external contour
    contours,_ = cv2.findContours(None,None,None)
    cnt = contours[0]

    img_contours = np.zeros(region.shape)
    # Draw the contours on the empty image
    cv2.drawContours(img_contours, contours, -1, (255,255,255), 1)
    plt.imshow(img_contours,cmap='gray')
    plt.show()

    # Compute area
    A = cv2.contourArea(None)

    # Compute perimeter
    P = cv2.arcLength(None,True)

    print("Area:", A)
    print("Perimeter:", P, '\n')

    # Compute circularity
    C = None

    return C

You can use the next code to **test if the results are right**:

In [ ]:
# Read the images
zero = cv2.imread(images_path + 'region_0.png', cv2.IMREAD_GRAYSCALE)
J = cv2.imread(images_path + 'region_J.png', cv2.IMREAD_GRAYSCALE)
B = cv2.imread(images_path + 'region_B.png', cv2.IMREAD_GRAYSCALE)
six = cv2.imread(images_path + 'region_6.png', cv2.IMREAD_GRAYSCALE)

# And show their circularity!
C_zero, C_J, C_B, C_six = circularity(zero), circularity(J), circularity(B), circularity(six)
print("Circularity of 0: ", round(C_zero,5))
print("Circularity of J: ", round(C_J,5))
print("Circularity of B: ", round(C_B,5))
print("Circularity of 6: ", round(C_six,5))

<font color='blue'>**Expected output:**  </font>

```
Circularity of 0:  0.82792
Circularity of J:  0.22803
Circularity of B:  0.7131
Circularity of 6:  0.36137
```

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Excellent! Now, **answer the following questions:**

- Why does `region_0.png` have the highest circularity?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Could we differentiate all the characters using only this feature as feature vector?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

- Is circularity invariant to translation, rotation or scale?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

## 6.1.2 Elongation <a id="612"></a>

Another simple shape descriptor is **elongation**, the ratio between the major and minor dimensions, $L_{max}$ and $L_{min}$, of the **rotated bounding box** of the region, that is, the smallest rectangle that contains the region, whatever its orientation (see Fig. 5):

$$E = \frac{L_{max}}{L_{min}}$$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/elongation.png" width="400"/>
    <figcaption>Fig. 5. Elongation is the ratio between the major and minor dimensions of the rotated bounding box.</figcaption>
</center>$\\[5pt]$

Compact shapes, like a circle or a square, have the lowest elongation, $E = 1$, and the more stretched the region, the higher its elongation.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Rotated bounding boxes</font>  
> OpenCV computes the rotated bounding box of a contour with [`cv2.minAreaRect()`](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#ga3d476a3417130ae5154aea421ca7ead9), so let's see how it works!

> **Suggested prompt**  
> *I need to compute the rotated bounding box of a contour with OpenCV's `cv2.minAreaRect()`. Do not write the full solution. Explain what this function returns and how it differs from `cv2.boundingRect()`.*

### **<span style="color:green"><b><i>ASSIGNMENT 2: Time to compute the elongation</i></b></span>**

Complete the function `elongation()`, which receives the `region` to be described as input and returns its elongation `E`, computing the rotated bounding box of its contour with `cv2.minAreaRect()`.

*Tip: note that the width of the rotated bounding box is not necessarily its major dimension.*

In [ ]:
# Assignment 2
def elongation(region):
    """ Compute the elongation of a region.

        Args:
            region: Binary image

        Returns:
            E: Elongation of the region (greater than or equal to 1)
    """

    # Get external contour
    contours,_ = cv2.findContours(None,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE)
    cnt = contours[0]

    # Get the rotated bounding box: center, (width, height) and angle
    _,(w,h),_ = cv2.minAreaRect(None)

    # Get its major and minor dimensions
    L_max = None
    L_min = None

    # Compute elongation
    E = None

    return E

You can use the next code to **test if the obtained results are correct**:

In [ ]:
# Read the images
zero = cv2.imread(images_path + 'region_0.png', cv2.IMREAD_GRAYSCALE)
J = cv2.imread(images_path + 'region_J.png', cv2.IMREAD_GRAYSCALE)
B = cv2.imread(images_path + 'region_B.png', cv2.IMREAD_GRAYSCALE)
six = cv2.imread(images_path + 'region_6.png', cv2.IMREAD_GRAYSCALE)

# And show their elongation!
print("Elongation of 0: ", round(elongation(zero),5))
print("Elongation of J: ", round(elongation(J),5))
print("Elongation of B: ", round(elongation(B),5))
print("Elongation of 6: ", round(elongation(six),5))

<font color='blue'>**Expected output:**  </font>

```
Elongation of 0:  1.7037
Elongation of J:  2.214
Elongation of B:  1.7037
Elongation of 6:  1.73585
```

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Now, **answer the following questions:**

- Why do `region_0.png` and `region_B.png` have the same elongation?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Is elongation invariant to translation, rotation or scale? Would it be if we used the upright bounding rectangle returned by `cv2.boundingRect()` instead of the rotated one?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

## 6.1.3 Building a feature vector <a id="613"></a>

Now that we can compute two different features, circularity ($f_1 = C$) and elongation ($f_2 = E$), we can build a feature vector $\mathbf{f}$ for characterizing each region by concatenating both features, that is, $\mathbf{f}=[f_1,f_2]^\top$.

Before sending to UMA our solution for region description, let's see if these features are discriminative enough to differentiate between the considered characters.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Plotting points with pyplot</font>  
> **Suggested prompt**  
> *I need to plot a 2D feature vector as a single point in a scatter plot with Matplotlib's `plt.plot()`. Do not write the full solution. Explain which arguments set the coordinates of the point, and how a format string like `'go'` sets its color and marker.*

### **<span style="color:green"><b><i>ASSIGNMENT 3: Plotting feature vectors</i></b></span>**

**Your task is** to plot the feature vectors, computed by the functions `circularity()` and `elongation()`, in a 2D space called the **<span style="color:DarkOrchid">feature space</span>**. In such a space, the **x-axis represents the circularity** $C$ of a region and the **y-axis its elongation** $E$.

In this way, if the descriptions of the considered characters don't appear close to each other in this space, they can be differentiated by relying on those features. **The problem appears if two or more characters have similar features** (their respective points are near). This tells us that **those features are just not enough** to automatically recognize the plate characters.

The plot of the first character is given as an example. Complete the `plt.plot()` calls of the rest, using the same color as their labels.

In [ ]:
# Assignment 3
matplotlib.rcParams['figure.figsize'] = (6.0, 6.0)

# Read the images
zero = cv2.imread(images_path + 'region_0.png', cv2.IMREAD_GRAYSCALE)
J = cv2.imread(images_path + 'region_J.png', cv2.IMREAD_GRAYSCALE)
B = cv2.imread(images_path + 'region_B.png', cv2.IMREAD_GRAYSCALE)
six = cv2.imread(images_path + 'region_6.png', cv2.IMREAD_GRAYSCALE)

# Build the feature vectors
f_zero = np.array([circularity(None), elongation(None)])
f_J = np.array([circularity(None), elongation(None)])
f_B = np.array([circularity(None), elongation(None)])
f_six = np.array([circularity(None), elongation(None)])

# Define the scatter plot
fig, ax = plt.subplots()
plt.axis([0, 1, 1, 2.5])
plt.xlabel("Circularity $C$")
plt.ylabel("Elongation $E$")

# Plot the points (the first one is given as an example)
plt.plot(f_zero[0], f_zero[1], 'go')
plt.text(f_zero[0]+0.02, f_zero[1]+0.05, '0', bbox={'facecolor': 'green', 'alpha': 0.5, 'pad': 10})
plt.plot(None, None, None)
plt.text(f_J[0]+0.02, f_J[1]+0.05, 'J', bbox={'facecolor': 'red', 'alpha': 0.5, 'pad': 10})
plt.plot(None, None, None)
plt.text(f_B[0]+0.02, f_B[1]+0.05, 'B', bbox={'facecolor': 'magenta', 'alpha': 0.5, 'pad': 10})
plt.plot(None, None, None)
plt.text(f_six[0]+0.02, f_six[1]+0.05, '6', bbox={'facecolor': 'blue', 'alpha': 0.5, 'pad': 10});

### <font color="blue"><b><i>Thinking about it (3)</i></b></font>

**What do you think?**

- Are these features discriminative enough?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- If your answer is no, how could we handle this problem?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Discriminative power of a feature vector</font>  
> **Suggested prompt**  
> *I have described four plate characters (0, J, B and 6) with the feature vector [circularity, elongation] and plotted them in a 2D feature space. Guide me, without giving me the answer, to reason whether these features are discriminative enough to recognize all the characters of a plate, and which kind of extra information could help.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

## 6.1.4 Moments <a id="614"></a>

Let's now try a more general family of shape descriptors: the **moments**, which summarize the spatial distribution of the pixels of the region. They are computed from its **binary mask** $M(x,y)$, which is $1$ for the pixels of the region and $0$ otherwise. The **geometric moment** of order $(p,q)$ is:

$$m_{pq} = \sum_x \sum_y x^p \, y^q \, M(x,y)$$

Some of them have a direct geometric meaning (see Fig. 6): the **area** is $A = m_{00}$, and the **centroid** is $\bar{x} = m_{10}/m_{00}$, $\bar{y} = m_{01}/m_{00}$.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/binary_moments.png" width="500"/>
    <figcaption>Fig. 6. Binary mask $M(x,y)$ of a region in image coordinates (the $y$ axis points down), its area $A = m_{00}$ and its centroid $(\bar{x},\bar{y})$.</figcaption>
</center>$\\[5pt]$

The geometric moments change when the region moves, grows or rotates, but they can be normalized step by step to obtain **invariant** descriptors:

1. **Translation**: the **central moments** $\mu_{pq} = \sum_x \sum_y (x-\bar{x})^p \, (y-\bar{y})^q \, M(x,y)$ are computed with respect to the centroid.
2. **Scale**: the **normalized moments** $\eta_{pq} = \mu_{pq} / \mu_{00}^{\,1+\frac{p+q}{2}}$ (with $p+q \geq 2$) also divide by the area.
3. **Rotation**: the normalized moments still change when the region rotates, so they have to be combined, which leads us to the Hu moments (next section).

## 6.1.5 Hu moments <a id="615"></a>

The **Hu moments** (proposed by Ming-Kuei Hu in 1962) are 7 combinations of the normalized moments $\eta_{pq}$ that don't change when the region rotates. The first two are:

$$h_1 = \eta_{20} + \eta_{02}, \qquad h_2 = (\eta_{20} - \eta_{02})^2 + 4\eta_{11}^2$$

and $h_3, \dots, h_7$ combine the third-order normalized moments ($p+q=3$). Together, they form a 7-D shape descriptor $\mathbf{h} = [h_1, \dots, h_7]$ that is **invariant to translation, scale and rotation** (see Fig. 7). Only $h_7$ changes its sign when the region is reflected, which allows telling mirror images apart.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/moments_invariance.gif" width="800"/>
    <figcaption>Fig. 7. Moments of a region while it is translated, scaled and rotated: the geometric moments $m_{pq}$ change, the normalized moments $\eta_{pq}$ only change with the rotation, and the Hu moments $\mathbf{h}$ remain the same.</figcaption>
</center>$\\[5pt]$

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Moments in OpenCV</font>  
> To compute the Hu moments, we first need the moments of the region. Fortunately, OpenCV implements their computation in [`cv2.moments()`](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#ga556a180f43cab22649c23ada36a8a139), which returns all of them ($m_{pq}$, $\mu_{pq}$ and $\eta_{pq}$ up to order 3) in a dictionary, so let's see how it works!

> **Suggested prompt**  
> *I need to compute the moments of a binary region with OpenCV's `cv2.moments()`. Do not write the full solution. Explain what the keys of the returned dictionary mean (`m`, `mu` and `nu`), and what the `binaryImage` parameter changes when the input is an image with values 0 and 255.*

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Hu moments in OpenCV</font>  
> OpenCV also implements the Hu moments in [`cv2.HuMoments()`](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#gab001db45c1f1af6cbdbe64df04c4e944), so let's see how it works!

> **Suggested prompt**  
> *I need to compute the Hu moments of a binary region with OpenCV's `cv2.HuMoments()`. Do not write the full solution. Explain what this function takes as input, what it returns (and its shape), and how it relates to the dictionary returned by `cv2.moments()`.*

### **<span style="color:green"><b><i>ASSIGNMENT 4: Describing plate characters with Hu moments</i></b></span>**

**What to do?** Complete the function `hu_moments()`, which computes the moments of the binary mask $M(x,y)$ of a region with `cv2.moments()` (using `binaryImage=True`), then computes its Hu moments from them with `cv2.HuMoments()`, and returns the Hu descriptor $\mathbf{h}$ as a 1-D array of 7 elements.

The Hu moments take values of very different magnitudes (e.g. $h_1$ is around $10^{-1}$, while $h_7$ can be smaller than $10^{-10}$), so they are usually compared in a **logarithmic scale**, $-\operatorname{sign}(h_i)\log_{10}|h_i|$, which keeps their sign. The function `log_scale()` below does this for you.

In [ ]:
# Assignment 4
def hu_moments(region):
    """ Compute the Hu moments of the region in a binary image.

        Args:
            region: Binary image

        Returns:
            h: array with the 7 Hu moments of the region
    """

    # Compute the moments of the binary mask M(x,y)
    moments = cv2.moments(None, binaryImage=None)

    # Compute the Hu moments and convert them to a 1-D array
    h = cv2.HuMoments(None).flatten()

    return h


def log_scale(h):
    """ Show the Hu moments in a logarithmic scale, keeping their sign. """
    return -np.sign(h) * np.log10(np.abs(h))

Let's use it to **describe our plate characters**. The next code prints the Hu descriptor $\mathbf{h}$ of each one (in logarithmic scale):

In [ ]:
np.set_printoptions(precision=3, suppress=True)

# Read the images
zero = cv2.imread(images_path + 'region_0.png', cv2.IMREAD_GRAYSCALE)
J = cv2.imread(images_path + 'region_J.png', cv2.IMREAD_GRAYSCALE)
B = cv2.imread(images_path + 'region_B.png', cv2.IMREAD_GRAYSCALE)
six = cv2.imread(images_path + 'region_6.png', cv2.IMREAD_GRAYSCALE)

for name, region in [('0', zero), ('J', J), ('B', B), ('6', six)]:
    print('h of ' + name + ':', log_scale(hu_moments(None)))

<font color='blue'>**Expected output:**  </font>

```
h of 0: [ 0.399  1.589  5.696  6.244 12.214  7.135 13.414]
h of J: [0.199 0.61  1.045 1.708 3.411 2.499 3.139]
h of B: [ 0.486  1.66   3.235  4.794 -8.871 -5.657  9.108]
h of 6: [ 0.449  1.519  2.897  3.546 -7.121 -4.767 -6.814]
```

### <font color="blue"><b><i>Thinking about it (4)</i></b></font>

Now, **answer the following questions:**

- Do the Hu descriptors of the four characters differ enough to tell them apart? Compare them with the feature vectors of Assignment 3.

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Which character is the most different from the rest? Does it agree with its shape?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

### **<span style="color:green"><b><i>ASSIGNMENT 5: Checking the invariance of the Hu moments</i></b></span>**

In the number-plate recognition problem, the descriptor has to be (at least) **translation and scale invariant**, because a car could stop closer to or further away from the camera, and in different positions.

**What to do?** Compute the Hu descriptor of the `J` region, of a rotated version of it (90 degrees), of a scaled version (by a factor of 2 in both axes) and of a mirrored version (flipped left-right). The code then compares them in a bar chart.

*Hint: You can rotate a NumPy array using [`np.rot90()`](https://numpy.org/doc/stable/reference/generated/numpy.rot90.html), mirror it with [`np.fliplr()`](https://numpy.org/doc/stable/reference/generated/numpy.fliplr.html), and scale an image using [`cv2.resize()`](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d).*

In [ ]:
# Assignment 5
matplotlib.rcParams['figure.figsize'] = (15.0, 5.0)

# Read the region and build its transformed versions
region = cv2.imread(images_path + 'region_J.png', cv2.IMREAD_GRAYSCALE)
versions = {
    'Original': region,
    'Rotated': np.rot90(None),
    'Scaled': cv2.resize(None, dsize=None, fx=None, fy=None, interpolation = cv2.INTER_AREA), # keep the dsize=None and the interpolation
    'Mirrored': np.fliplr(None),
}

# Show the regions
for k, (name, image) in enumerate(versions.items()):
    plt.subplot(1, 4, k+1)
    plt.imshow(image, cmap='gray')
    plt.title(name)
plt.show()

# Compute and compare their Hu descriptors
labels = np.arange(1, 8)
for k, (name, image) in enumerate(versions.items()):
    h = log_scale(hu_moments(None))
    plt.bar(labels + 0.2*(k-1.5), h, width=0.2, label=name)
    print(name.ljust(9), h)
plt.xticks(labels, ['$h_' + str(i) + '$' for i in labels])
plt.legend();

### <font color="blue"><b><i>Thinking about it (5)</i></b></font>

Now, **answer the following questions:**

- What happens with the mirrored version? Could this be useful (or a problem) for recognizing plate characters?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Now that you can describe a binary region in different ways, **what descriptor would you use** for the UMA parking problem? **Why?**

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Choosing a shape descriptor</font>  
> **Suggested prompt**  
> *I have compared circularity, elongation and the Hu moments to describe plate characters that may appear translated and scaled in the image. Guide me, without deciding for me, to reason which descriptor I should use for this problem and which limitations it may have (e.g. noise in the segmentation, or characters that are mirror images of each other).*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

Surf the internet looking for **more shape features**, and try to find a pair of them working better than circularity and elongation.

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

Take an image of a car plate, apply the techniques already studied in the course to improve its quality, and binarize it. Then, extract some shape features and check where the numbers/letters are projected in the feature space.

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

## Conclusion

Great work! You have learned about:

- the aim of region descriptors: representing a region with a compact feature vector $\mathbf{f}$,
- two simple shape descriptors, circularity and elongation, and how to build a feature vector with them and analyze its discriminative power, and
- the moments of a binary region, and how the central, normalized and Hu moments achieve invariance to translation, scale and rotation.

In the next notebook, we will describe regions by their appearance: intensity statistics and texture!

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>